# Healthcare Fraud Detection Demo

This notebook demonstrates how new claim files are automatically ingested using **Databricks Delta Live Tables (DLT) with Auto Loader**.

The pipeline processes data through the **Medallion Architecture**:

Bronze → Raw ingestion  
Silver → Cleaned and standardized data  
Gold → Fraud detection analytics

In this demo we simulate two scenarios:

1. **High Risk Claim**
2. **Normal Claim**

The fraud engine evaluates risk signals such as:
- High claim amount
- Payment anomaly
- Procedure charge mismatch
- High risk provider

## Step 1 - Verify Existing Claims

Before uploading a new file, we confirm that the claim does not exist in the system.

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.prod_schema_hc.fact_fraud_claims
WHERE claim_id IN (54001,54002);
--Currently the system has no records for these claims.

## Step 2 - Upload High Risk Claim

A new claim file is uploaded to the monitored storage location.

Auto Loader automatically detects the file and processes it through the pipeline.

Claim Details:
- Claim ID: 54001
- Claim Amount: 95,000

## Step 3 - Fraud Engine Evaluation

The fraud engine evaluates risk signals and calculates a fraud score.

In [0]:
%sql
SELECT
claim_id,
claim_amount,
fraud_score,
fraud_flag,
fraud_severity
FROM dlt_hc_fraud_detection.prod_schema_hc.fact_fraud_claims
WHERE claim_id = 54001;

--Since the claim amount exceeds the configured risk threshold,the fraud engine increases the risk score and flags the claim for review.

## Step 4 - Fraud Rule Breakdown

To make the fraud decision transparent, we expose the risk signals evaluated by the fraud engine.

In [0]:
%sql
SELECT
claim_id,
claim_amount,

CASE
WHEN claim_amount >= 75000 THEN 'YES'
ELSE 'NO'
END AS high_claim_signal,

fraud_score,
fraud_flag

FROM dlt_hc_fraud_detection.prod_schema_hc.fact_fraud_claims
WHERE claim_id = 54001;
--The high claim signal was triggered, which increased the fraud risk score.

## Step 5 - Upload Normal Claim

Now we upload a normal claim with a low claim amount.

Claim Details:
- Claim ID: 54002
- Claim Amount: 8,000

## Step 6 - Fraud Engine Result

Since this claim does not trigger any risk signals, it is classified as a low risk claim.

In [0]:
%sql
SELECT
claim_id,
claim_amount,
fraud_score,
fraud_flag,
fraud_severity
FROM dlt_hc_fraud_detection.prod_schema_hc.fact_fraud_claims
WHERE claim_id = 54002;
--Since none of the fraud signals are triggered, the claim is considered safe.

## Fraud Risk Rule Engine Logic

The fraud detection engine evaluates multiple risk signals and assigns a risk score to each claim.

Each rule contributes a specific number of points.

Fraud Decision Logic:

| Rule | Condition | Points |
|-----|-----------|-------|
High Claim Amount | claim_amount ≥ 75,000 | 20 |
Payment Anomaly | paid_amount > claim_amount | 40 |
Procedure Mismatch | total_procedure_charge > claim_amount | 20 |
High Risk Provider | provider risk_score ≥ 0.75 | 30 |

### Final Fraud Decision

| Fraud Score | Decision |
|-------------|---------|
0 – 39 | Normal Claim |
40 – 69 | Review Required |
70+ | High Fraud Risk |

## Fraud Investigation Dashboard

This section provides a transparent view of the fraud evaluation process.

For each claim we display:
- Risk signals evaluated by the fraud engine
- Individual rule outcomes
- Final fraud score
- Final fraud decision

This helps investigators understand why a claim was flagged.

In [0]:
%sql
SELECT
claim_id,
provider_id,
patient_id,
claim_amount,
paid_amount,
total_procedure_charge,

-- Risk Signals
CASE 
WHEN claim_amount >= 75000 
THEN 'Triggered'
ELSE 'Not Triggered'
END AS high_claim_rule,

CASE 
WHEN paid_amount > claim_amount 
THEN 'Triggered'
ELSE 'Not Triggered'
END AS payment_anomaly_rule,

CASE 
WHEN total_procedure_charge > claim_amount 
THEN 'Triggered'
ELSE 'Not Triggered'
END AS procedure_mismatch_rule,

-- Fraud Decision
fraud_score,
fraud_flag,
fraud_severity

FROM dlt_hc_fraud_detection.prod_schema_hc.fact_fraud_claims
WHERE claim_id IN (54001,54002);

## Demo Conclusion

In this demonstration we simulated two healthcare claim scenarios to observe how the pipeline processes new data automatically.

**Claim 54001 - High Value Claim**
- Claim Amount: 95,000
- The fraud engine identified this as a higher-risk claim because the claim amount crossed the configured threshold.
- As a result, the fraud score increased and the claim was flagged for further review.

**Claim 54002 - Normal Claim**
- Claim Amount: 8,000
- This claim did not trigger any fraud risk signals.
- The fraud engine classified it as a normal claim with low risk.

### Key Takeaway

This demo highlights how **Databricks Delta Live Tables with Auto Loader** enables:

- Automatic detection of new files
- Incremental data ingestion
- Real-time transformation through the Medallion Architecture
- Automated fraud risk evaluation in the Gold layer

This architecture allows organizations to build **scalable, automated, and transparent data pipelines for fraud analytics**.